[<img src="https://gitlab.irit.fr/toc/etu-n7/controle-optimal/-/raw/master/ressources/Logo-toulouse-inp-N7.png" alt="N7" height="80"/>](https://gitlab.irit.fr/toc/etu-n7/controle-optimal)
<img src="https://gitlab.irit.fr/toc/ens-n7/texCoursN7/-/raw/main/logo-insa.png" alt="INSA" height="80" style="margin-left:50px"/>

# Calcul de dérivées

- Date : 2025-2026
- Durée approximative : 2h

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Introduction
</div>

Il existe plusieurs façons de calculer une dérivée sur un calculateur :

- par différences finies (schémas avant et centré) ;
- par différentiation complexe (pas imaginaire) ;
- en utilisant la différentiation automatique (nombres duaux, `ForwardDiff`) ;
- en utilisant le calcul formel et un générateur de code.

Nous étudions ici les trois premières familles, que nous appliquerons ensuite au
calcul des équations variationnelles (ou linéarisées) des équations différentielles.

On notera $\|\cdot\|$ la norme euclidienne usuelle et $\mathcal{N}(0,1)$ une variable
aléatoire gaussienne centrée réduite.

In [ ]:
# activation du projet situé dans le répertoire de ce notebook
using Pkg
Pkg.activate(@__DIR__)

# chargement des paquets
using DualNumbers
using ForwardDiff
using LinearAlgebra
using Plots
using Plots.Measures
using Polynomials
using Printf

# figures vectorielles (nettes à tout zoom)
default(fmt = :svg)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Dérivées par différences finies avant
</div>

Soit $f$ une fonction lisse de $\mathbb{R}^{n}$ dans $\mathbb{R}^{m}$, $x$ un point de
$\mathbb{R}^{n}$ et $v$ un vecteur de $\mathbb{R}^{n}$. On pose $g \colon h \mapsto f(x+hv)$,
de sorte que
$$
    g'(0) = f'(x)\cdot v, \qquad g^{(k)}(0) = f^{(k)}(x)(v,\dots,v).
$$
D'après la formule de Taylor-Young à l'ordre $p$ (on choisit $p$ indépendamment de la
dimension $n$) :
$$
    g(h) = \sum_{i=0}^{p} \frac{h^i}{i!}\, g^{(i)}(0) + R_p(h), \qquad R_p(h) = o(h^p),
$$
et d'après l'**inégalité de Taylor-Lagrange** (polycopié, formules de Taylor), pour $h > 0$,
$$
    \| R_p(h) \| \leq \frac{M_p\, |h|^{p+1}}{(p+1)!},
    \qquad M_p := \sup_{[0,h]} \big\| g^{(p+1)} \big\|.
$$
On écrit une **majoration**, et non une égalité
$R_p(h) = \frac{h^{p+1}}{(p+1)!}\, g^{(p+1)}(\xi)$ pour un $\xi \in {]0,h[}$ : cette forme avec
point intermédiaire, valable pour $g$ à valeurs réelles, tombe en défaut dès que $g$ est à
valeurs dans $\mathbb{R}^m$, $m \geq 2$. Déjà pour $p = 0$, l'égalité des accroissements finis
$g(h) - g(0) = h\, g'(\xi)$ y est fausse — contre-exemple du polycopié :
$x \mapsto (\cos 2\pi x,\, \sin 2\pi x)$, dont la dérivée garde une norme constante alors que
$g(1) - g(0) = 0$.

La méthode des différences finies avant approche $f'(x)\cdot v$ par
$$
    \frac{f(x+hv) - f(x)}{h} =
    \frac{g(h)-g(0)}{h} = g'(0) + \frac{h}{2}\, g^{(2)}(0) + \frac{h^2}{6}\, g^{(3)}(0) + o(h^2).
$$
L'approximation est d'ordre 1 si $g^{(2)}(0) \neq 0$, au moins d'ordre 2 sinon.

**Remarque (epsilon machine).** Sur machine, les calculs se font en virgule flottante.
On note $\mathrm{eps}_\mathrm{mach}$ le plus petit nombre tel que
$1+\mathrm{eps}_\mathrm{mach}\ne 1$. Cette quantité dépend de la machine et de l'encodage.
En `Julia` : `eps()` (ou `eps(Float64)`, `eps(Float32)`).

Notons $\mathrm{num}(g,h)$ la valeur de $g(h)$ calculée numériquement. On suppose l'erreur
**absolue** d'évaluation majorée par
$$
  \left\| \mathrm{num}(g,h) - g(h) \right\| =: \| e_h\| \leq \mathrm{eps}_\mathrm{mach}\, L_f,
$$
où $L_f$ est de l'ordre de l'amplitude de $f$ sur le domaine d'intérêt : elle absorbe
l'échelle des valeurs de $f$ et, si $\|x\|$ est grand, l'erreur de représentation de
$x+hv$. Alors
\begin{align*}
    \left\| \frac{\mathrm{num}(g,h) - \mathrm{num}(g,0)}{h} - g'(0) \right\|
    &= \left\| \frac{g(h) + e_h - g(0) - e_0}{h} - g'(0) \right\| \\[0.8em]
    &= \left\| \frac{R_1(h)}{h} + \frac{e_h - e_0}{h} \right\| \\[0.8em]
    &\leq \left\| \frac{R_1(h)}{h} \right\| + \left\| \frac{e_h - e_0}{h} \right\| \\[0.8em]
    &\leq \underbrace{\frac{M_1\, h}{2}}_{\text{erreur de troncature}}
        + \underbrace{\frac{2\, \mathrm{eps}_\mathrm{mach}\, L_f}{h}}_{\text{erreur d'arrondi}}.
\end{align*}
Ce majorant atteint son minimum en
$$
    h_{*} = 2\, \sqrt{\frac{\mathrm{eps}_\mathrm{mach}\, L_f}{M_1}},
    \qquad \text{de valeur} \qquad
    \varphi(h_*) = 2\sqrt{M_1\, L_f\, \mathrm{eps}_\mathrm{mach}}.
$$
En supposant le problème bien mis à l'échelle ($L_f \simeq M_1 \simeq 1$), le choix le plus
**judicieux** est
$$
    h_{*} \approx 2\sqrt{\mathrm{eps}_\mathrm{mach}} \;\sim\; \sqrt{\mathrm{eps}_\mathrm{mach}},
$$
et la précision atteignable n'est alors que
$$
    \varphi(h_*) \approx 2\sqrt{\mathrm{eps}_\mathrm{mach}} \approx 3\times 10^{-8}
    \quad (\text{Float64}),
$$
soit **la moitié des chiffres significatifs**. C'est cette perte qui motive les schémas
suivants.

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Dérivées par différences finies centrées
</div>

En reprenant $g \colon h \mapsto f(x+hv)$, on a aussi
$$
    g(-h) = \sum_{i=0}^{p} \frac{(-h)^i}{i!}\, g^{(i)}(0) + \tilde R_p(h),
    \qquad \tilde R_p(h) = o(h^p).
$$
Le schéma centré approche $f'(x)\cdot v$ par
$$
    \frac{f(x+hv) - f(x-hv)}{2h} = \frac{g(h) - g(-h)}{2h} =
    g'(0) + \frac{h^2}{6}\, g^{(3)}(0) + \mathcal{O}(h^4).
$$
Les termes pairs se compensent : l'approximation est d'ordre 2 si $g^{(3)}(0) \neq 0$,
au moins d'ordre 4 sinon. Le schéma coûte une évaluation de $f$ de plus que le schéma avant.

L'analyse de l'erreur est identique à celle du schéma avant (**exercice 3, en fin de TP**)
et conduit à
$$
    \psi(h) \leq \underbrace{\frac{M_2}{6}\, h^2}_{\text{troncature}}
              + \underbrace{\frac{\mathrm{eps}_\mathrm{mach}\, L_f}{h}}_{\text{arrondi}},
    \qquad M_2 := \sup \big\| g^{(3)} \big\|,
$$
minimal en
$$
    h_{*} = \left( \frac{3\, \mathrm{eps}_\mathrm{mach}\, L_f}{M_2} \right)^{1/3}
          \approx \sqrt[3]{3\, \mathrm{eps}_\mathrm{mach}}
          \approx 1.44\; \mathrm{eps}_\mathrm{mach}^{1/3},
$$
pour une précision de l'ordre de $\mathrm{eps}_\mathrm{mach}^{2/3} \approx 4\times 10^{-11}$
(Float64), soit les deux tiers des chiffres significatifs. Le facteur devant la racine
cubique est $\sqrt[3]{3}$, et non $2$ comme pour le schéma avant.

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Dérivées par différentiation complexe
</div>

Les schémas avant et centré forment une différence $\Delta f = f(x+hv) - f(x)$ de deux
quantités voisines : quand $h \to 0$, la **soustraction** perd des chiffres significatifs
(*cancellation*), d'où le terme d'arrondi en $1/h$ et le compromis sur $h$. Les
[différences à pas complexe](https://dl.acm.org/doi/10.1145/838250.838251) suppriment
cette soustraction.

On suppose $f$ **holomorphe** au voisinage de $x$ (dérivable au sens complexe : une seule
dérivée quelle que soit la direction d'approche dans $\mathbb{C}$). Avec un pas imaginaire
$ih$ :
$$
    f(x+ihv) = g(ih) = g(0) + ih\, g'(0) - \frac{h^2}{2}\, g^{(2)}(0)
             - i\frac{h^3}{6}\, g^{(3)}(0) + o(h^3).
$$
La partie imaginaire donne directement, **sans aucune soustraction** :
$$
    f'(x)\cdot v = g'(0) \approx \frac{\mathrm{Im}\big(f(x+ihv)\big)}{h}
    = g'(0) - \frac{h^2}{6}\, g^{(3)}(0) + \mathcal{O}(h^4).
$$
L'approximation est d'ordre 2. Le terme d'ordre 0 et les termes pairs vivent dans la
partie réelle, séparément.

**Pas de compromis sur $h$.** La quantité calculée, $\mathrm{Im}(f(x+ihv))$, est elle-même
d'ordre $h\,\|f'\|$ et obtenue avec une erreur *relative* $\sim \mathrm{eps}_\mathrm{mach}$,
donc $\|e_h\| \lesssim \mathrm{eps}_\mathrm{mach}\, C\, h\, \|f'\|$ et
$$
    \left\| \frac{\mathrm{num}(g,ih)}{h} - g'(0) \right\|
    \lesssim \underbrace{\frac{M_2}{6}\, h^2}_{\to\, 0}
           + \underbrace{\mathrm{eps}_\mathrm{mach}\, C\, \|f'\|}_{\text{constant en } h}.
$$
L'erreur décroît en $h^2$ puis **atteint un plancher $\sim \mathrm{eps}_\mathrm{mach}$ et y
reste** — elle ne remonte pas. Tout pas
$$
    h \leq h_{*} \approx \sqrt{\mathrm{eps}_\mathrm{mach}}
$$
convient, et l'on peut descendre bien plus bas (jusqu'à l'underflow, $h \sim 10^{-150}$).
L'argument précis — *pourquoi la partie réelle ne pollue pas la partie imaginaire* — est
donné en **annexe A**.

**Remarque (`Julia`).** `im` est l'unité imaginaire, `imag(z)` la partie imaginaire. La
méthode exige que tout le code de $f$ soit bâti sur des opérations holomorphes : `abs`,
`min`, `max`, les comparaisons et tout code par morceaux la cassent (voir « Limites » en
fin de TP).

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Dérivées par différentiation automatique via les nombres duaux
</div>

**L'algèbre.** Un nombre dual s'écrit $a + b\,\varepsilon$ avec $(a,b)\in\mathbb{R}^2$,
$\varepsilon \neq 0$ et $\varepsilon^2 = 0$ : c'est l'anneau commutatif
$\mathbb{R}[\varepsilon]/(\varepsilon^2)$. Les opérations se calculent en développant et en
supprimant $\varepsilon^2$ :
$$
\begin{aligned}
(a + b\varepsilon) + (c + d\varepsilon) &= (a+c) + (b+d)\,\varepsilon, \\[0.4em]
(a + b\varepsilon)\,(c + d\varepsilon)
   &= ac + (ad + bc)\,\varepsilon + \underbrace{bd\,\varepsilon^2}_{=\,0}
    = ac + (ad + bc)\,\varepsilon, \\[0.4em]
\frac{1}{a + b\varepsilon} &= \frac{1}{a} - \frac{b}{a^2}\,\varepsilon \qquad (a \neq 0).
\end{aligned}
$$

**Le lien avec la dérivation.** Pour $f \colon \mathbb{R}\to\mathbb{R}$ dérivable, on
*définit* l'action de $f$ sur les duaux par la convention
$$
    f(a + b\,\varepsilon) := f(a) + f'(a)\, b\, \varepsilon.
$$
La partie duale de $f(x + \varepsilon)$ (cas $b=1$) est alors exactement $f'(x)$. Surtout,
cette convention est **compatible avec toute la structure d'anneau** — c'est là toute la
magie. Posons $d = x + \varepsilon$ :

- **Somme.**
  $(f+g)(d) = \big(f(x) + f'(x)\varepsilon\big) + \big(g(x) + g'(x)\varepsilon\big)
   = (f+g)(x) + (f+g)'(x)\,\varepsilon.$

- **Produit.**
  $(fg)(d) = \big(f(x)+f'(x)\varepsilon\big)\big(g(x)+g'(x)\varepsilon\big)
   = f(x)g(x) + \big(f'(x)g(x) + f(x)g'(x)\big)\varepsilon
   = (fg)(x) + (fg)'(x)\,\varepsilon$ :
  la règle du produit tombe du terme $ad+bc$.

- **Composée.**
  $(g\circ f)(d) = g\big(f(x) + f'(x)\varepsilon\big)
   = g\big(f(x)\big) + g'\big(f(x)\big)\,f'(x)\,\varepsilon
   = (g\circ f)(x) + (g\circ f)'(x)\,\varepsilon$,
  en appliquant la convention à $g$ avec partie duale $f'(x)$ : c'est la dérivation en
  chaîne.

**Conséquence — comment coder un outil de différentiation automatique.** Il « suffit » de

1. coder l'arithmétique de $\mathbb{R}[\varepsilon]/(\varepsilon^2)$
   ($+$, $-$, $\times$, $/$) ;
2. surcharger chaque fonction élémentaire (`sin`, `cos`, `exp`, `log`, `^`, …) via la
   convention $u \mapsto f(\mathrm{re}\,u) + f'(\mathrm{re}\,u)\,(\mathrm{dual}\,u)\,\varepsilon$.

Toute fonction obtenue par composition de ces briques est alors dérivée **exactement**
(à l'arrondi près, sans pas $h$) par simple propagation :
$\mathrm{dual}\big(f(x + \varepsilon)\big) = f'(x)$. C'est le principe de `ForwardDiff.jl`
(en plus optimisé, avec un $\varepsilon$ vectoriel pour obtenir le gradient d'un coup).

**Limite.** $\varepsilon^2 = 0$ efface l'ordre 2 : les nombres duaux ne donnent pas $f''$
directement — il faut les emboîter, ou utiliser des nombres hyper-duaux (voir « Dérivée
seconde » en fin de TP).

**En `Julia`** (paquet `DualNumbers` ; les raccourcis `real` / `dual` sont définis dans
les fonctions auxiliaires) :

```julia
using DualNumbers

ε = Dual(0.0, 1.0)               # 0 + 1ε, le nombre dual canonique

# cas scalaire
d = 1 + 2ε                       # ou 1 + 2*ε, ou 1 + ε*2
realpart(d)                      # 1.0
dualpart(d)                      # 2.0

# cas vectoriel
d = [1, 3] + [2, 4]ε             # ou [1 + 2ε, 3 + 4ε]
realpart.(d)                     # [1.0, 3.0]
dualpart.(d)                     # [2.0, 4.0]
```

**Remarque.** Le paquet `ForwardDiff` calcule les mêmes dérivées, plus efficacement que
`DualNumbers`.

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Fonctions auxiliaires
</div>

In [ ]:
# liste des méthodes disponibles
methods = (:forward, :central, :complex, :dual, :forward_ad)

"""
    mytypeof(x) -> Type

Type flottant de `x` (ou de ses coordonnées si `x` est un vecteur).
"""
function mytypeof(x::Union{T, Vector{<:T}}) where {T<:AbstractFloat}
    return T
end

"""
    _step(x, v, method) -> Real

Pas `h` par défaut d'un schéma à différences finies, issu de l'analyse d'erreur :
`√(eps)` (avant, complexe), `∛(eps)` (centré), remis à l'échelle par `√(‖x‖/‖v‖)`.
Vaut `0` pour `:dual` et `:forward_ad` (pas de pas).
"""
function _step(x, v, method)
    T = mytypeof(x)
    eps_value = eps(T)
    if method == :forward
        step = √(eps_value)
    elseif method == :central
        step = cbrt(eps_value)
    elseif method == :complex
        step = √(eps_value)
    else
        step = zero(T)
    end
    step *= √(max(one(T), T(norm(x)))) / √(max(one(T), T(norm(v))))
    return step
end

"Méthode utilisée par défaut par `derivative`."
_method() = :forward

# nombre dual canonique  ε = 0 + 1ε
ε = Dual(0.0, 1.0)

# raccourcis parties réelle / duale, y compris sur des tableaux
dual(x::Dual) = dualpart(x)
dual(x::AbstractArray{<:Dual}) = dualpart.(x)
real(x::Dual) = realpart(x)            # NB : masque volontairement Base.real sur Dual
real(x::AbstractArray{<:Dual}) = realpart.(x)

"""
    error_model(method; M=1.0, L=1.0, ϵ=eps()) -> (troncature, arrondi, h_star)

Majorant théorique de l'erreur du schéma `method` : `troncature(h)` et `arrondi(h)`
sont les deux contributions du majorant, `h_star` minimise leur somme.

- `M` : borne sur `‖g^{(p+1)}(0)‖`, `g : h ↦ f(x+hv)`, avec `p+1 = 2` (`:forward`)
  ou `3` (`:central`, `:complex`) ; terme de troncature.
- `L` : échelle de l'erreur d'arrondi sur `f`, `‖e_h‖ ≲ ϵ L`.
- `ϵ` : epsilon machine.

Pour `:complex`, il n'y a pas de soustraction : `arrondi` ne dépend pas de `h`, et
`h_star` est le pas en deçà duquel on a atteint le plancher `ϵ L`.
"""
function error_model(method; M=1.0, L=1.0, ϵ=eps())
    if method === :forward
        return (h -> M/2 * h,   h -> 2ϵ*L / h, 2*sqrt(ϵ*L/M))
    elseif method === :central
        return (h -> M/6 * h^2, h -> ϵ*L / h,  cbrt(3ϵ*L/M))
    elseif method === :complex
        return (h -> M/6 * h^2, h -> ϵ*L,      sqrt(6ϵ*L/M))
    else
        return (h -> zero(h),   h -> ϵ*L,      NaN)
    end
end

"""
    _estimate_M(f, x, v, k) -> Real

Estimation numérique de `‖g^{(k)}(0)‖` avec `g(t) = f(x + t v)`, par différences finies
centrées (`k = 2` ou `k = 3`). Sert à instancier `error_model`.
"""
function _estimate_M(f, x, v, k)
    g(t) = f(x .+ t .* v)
    if k == 2
        h = 1e-4
        d = (g(h) .- 2 .* g(zero(h)) .+ g(-h)) ./ h^2
    else
        h = 1e-3
        d = (g(2h) .- 2 .* g(h) .+ 2 .* g(-h) .- g(-2h)) ./ (2h^3)
    end
    return max(norm(d), 1e-12)
end

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
La méthode principale pour le calcul de dérivées
</div>

La fonction `derivative` ci-dessous calcule la dérivée directionnelle $f'(x)\cdot v$.

In [ ]:
"""
    derivative(f, x, v; method=_method(), h=_step(x, v, method))

Approximation de la dérivée directionnelle `f'(x)·v` par `method` :

| `method`      | principe                          | ordre | pas `h`            |
|:--------------|:----------------------------------|:------|:-------------------|
| `:forward`    | différences finies avant          | 1     | `≈ 2√eps`          |
| `:central`    | différences finies centrées       | 2     | `≈ ∛(3 eps)`       |
| `:complex`    | pas imaginaire (`f` holomorphe)   | 2     | `≤ √eps`, plancher qui ne remonte pas |
| `:dual`       | nombres duaux (AD, mode direct)   | exact | —                  |
| `:forward_ad` | idem via `ForwardDiff.jl`         | exact | —                  |

`v` est la direction ; `h` est ignoré pour `:dual` et `:forward_ad`.
"""
function derivative(f, x, v; method=_method(), h=_step(x, v, method))
    if method ∉ methods
        error("Choisir une méthode valide parmi ", methods)
    end
    if method == :forward
        return zero(f(x))                       # à compléter
    elseif method == :central
        return zero(f(x))                       # à compléter
    elseif method == :complex
        return zero(f(x))                       # à compléter
    elseif method == :dual
        return zero(f(x))                       # à compléter
    elseif method == :forward_ad
        if x isa Number
            return ForwardDiff.derivative(f, x) * v
        else
            return ForwardDiff.jacobian(f, x) * v
        end
    end
end;

In [ ]:
"""
    print_derivatives(f, x, v, exact)

Affiche, pour chaque méthode de `methods`, la valeur de `derivative`, l'erreur par
rapport à `exact` (= `f'(x)·v` calculé à la main) et, pour les schémas à pas, le pas
utilisé et le rapport erreur/pas.
"""
function print_derivatives(f, x, v, exact)

    println("Dérivée exacte : ", exact, "\n")

    for method ∈ methods
        dfv = derivative(f, x, v, method=method)
        println("Méthode : ", method)
        println("   dérivée  : ", dfv)
        @printf("   erreur   : %.3e\n", norm(dfv .- exact))
        if method ∈ (:forward, :central, :complex)
            h = _step(x, v, method)
            @printf("   pas h    : %.3e\n", h)
            @printf("   erreur/h : %.3e\n", norm(dfv .- exact) / h)
        end
        println()
    end

end;

### Exercice 1

1. Compléter la fonction `derivative` avec les schémas de différences finies avant et
   centré, la différentiation complexe et la différentiation automatique via les nombres
   duaux (le cas `:forward_ad` est déjà fourni).
2. Exécuter les deux cellules ci-dessous (cas scalaire et cas vectoriel) et vérifier les
   résultats.

In [ ]:
# cas scalaire :  f : ℝ → ℝ
f(x) = cos(x)
x    = π/4
v    = 1.0

exact = -sin(x) * v          # f'(x)·v à la main

print_derivatives(f, x, v, exact)

In [ ]:
# cas vectoriel :  f : ℝ² → ℝ²
f(x) = [0.5 * (x[1]^2 + x[2]^2); x[1] * x[2]]
x    = [1.0, 2.0]
v    = [1.0, -1.0]

exact = [x[1]*v[1] + x[2]*v[2], x[1]*v[2] + x[2]*v[1]]     # J_f(x)·v à la main

print_derivatives(f, x, v, exact)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Pas optimal
</div>

On étudie l'erreur des différentes méthodes en fonction du pas $h$, pour :

- $\cos$ au point $x_0 = \pi/3$ — cas bien mis à l'échelle ;
- $\cos$ au point $x_1 = 10^6 \times \pi/3$ — grand argument : l'erreur de représentation
  de $x_1$ domine, donc $L_f \sim \|x_1\|$ ;
- $\cos + 10^{-8}\,\mathcal{N}(0,1)$ au point $x_0$ — évaluation bruitée : $L_f$ est grand.

On prend $h = 10^{-i}$, $i \in \{1,\dots,16\}$, en échelle logarithmique sur les deux axes.

### Exercice 2

1. Avec `analyse`, tracer pour `:forward`, `:central`, `:complex` l'erreur en fonction de
   $h$, la borne théorique et le pas $h_*$. Comparer $h_*$ théorique et observé, et
   l'ordre observé à l'ordre attendu. Commentaires.
2. Avec `comparer`, superposer les cinq méthodes sur une même figure. Que dire de
   `:complex`, `:dual`, `:forward_ad` face aux différences finies ?
3. Reprendre pour $x_1$ et pour la fonction bruitée. Puis reprendre en précision
   `Float32`. Comment se déplacent les courbes et les $h_*$ ?

In [ ]:
"""
    analyse(method; f, dfv, x, v, precision=Float64,
            hs = 10.0 .^ range(-1, -16, length=120),
            L=nothing, M=nothing, plt=nothing, showbound=true, label=string(method))

Étudie le schéma `method` pour l'approximation de `f'(x)·v` (valeur exacte `dfv`) :

- trace l'erreur de `derivative` en fonction du pas `h` ;
- superpose le majorant théorique `troncature + arrondi` et le pas `h_star`
  (voir `error_model`), sauf si `showbound=false` ;
- estime l'ordre observé par régression linéaire de `log10(erreur)` sur `log10(h)`
  dans le régime pré-plancher (via `Polynomials.fit`).

Renvoie `(plt, h_star, ordre_observe)`. Pour `:dual` et `:forward_ad`, l'erreur ne
dépend pas de `h` : elle est tracée en trait horizontal et `ordre_observe` vaut `NaN`.
`M` et `L` sont estimés automatiquement si non fournis.
"""
function analyse(method; f, dfv, x, v,
                 precision::Type = Float64,
                 hs = 10.0 .^ range(-1, -16, length = 120),
                 L = nothing, M = nothing,
                 plt = nothing, showbound::Bool = true, label = string(method))

    ϵ  = eps(precision)
    xT = x isa Number ? convert(precision, x) : convert.(precision, x)
    vT = v isa Number ? convert(precision, v) : convert.(precision, v)

    # méthodes exactes : erreur indépendante de h -> trait horizontal
    # (segment explicite plutôt que hline!, pour rester compatible tous backends)
    if method ∈ (:dual, :forward_ad)
        d   = derivative(f, xT, vT; method = method)
        err = norm(d .- dfv)
        plt = plt === nothing ? plot(xscale = :log10, yscale = :log10) : plt
        c   = max(err, 1e-17)
        plot!(plt, [minimum(hs), maximum(hs)], [c, c], ls = :dot, lw = 2, label = label)
        @printf("%-11s  erreur ≈ %.2e   (indépendante de h)\n", label, err)
        return plt, NaN, NaN
    end

    k  = method === :forward ? 2 : 3
    Mv = M === nothing ? _estimate_M(f, xT, vT, k) : M
    Lv = L === nothing ? max(norm(f(xT)), norm(xT), one(ϵ)) : L
    trunc, round, hstar = error_model(method; M = Mv, L = Lv, ϵ = ϵ)

    errs = zeros(float(precision), length(hs))
    for (i, h) in pairs(hs)
        d = derivative(f, xT, vT; method = method, h = convert(precision, h))
        errs[i] = norm(d .- dfv)
    end

    # ordre observé : régression sur le régime où l'erreur décroît encore
    hsv  = collect(hs)
    kcut = findfirst(<(hstar), hsv)
    kcut = kcut === nothing ? length(hsv) : kcut
    kfit = min(kcut - 1, argmin(errs))
    good = [i for i in 1:max(kfit, 2) if errs[i] > 0]
    span = length(good) ≥ 2 ? log10(maximum(errs[good]) / minimum(errs[good])) : 0.0
    p_obs = span ≥ 1 ? Polynomials.fit(log10.(hsv[good]), log10.(errs[good]), 1)[1] : NaN

    plt = plt === nothing ? plot(xscale = :log10, yscale = :log10) : plt
    plot!(plt, hsv, max.(errs, 1e-18), marker = :circle, ms = 2,
          markerstrokewidth = 0, lw = 1.5, label = label)
    if showbound
        hb = 10.0 .^ range(log10(minimum(hsv)), log10(maximum(hsv)), length = 200)
        bound = trunc.(hb) .+ round.(hb)
        plot!(plt, hb, bound, ls = :dash, lw = 1, label = "majorant")
        ylo, yhi = extrema(vcat(max.(errs, 1e-18), bound))
        plot!(plt, [hstar, hstar], [ylo, yhi], ls = :dashdot, lw = 1, label = "h*")
    end
    plot!(plt, xlabel = "h", ylabel = "erreur", legend = :outertopright, bottom_margin = 5mm)

    emin = isempty(good) ? minimum(errs) : minimum(errs[errs .> 0])
    @printf("%-11s  h* théo = %.2e   erreur min ≈ %.2e   ordre observé ≈ %s\n",
            label, hstar, emin, isnan(p_obs) ? "n/a" : @sprintf("%.2f", p_obs))
    return plt, hstar, p_obs
end

"""
    comparer(; f, dfv, x, v, precision=Float64, hs=...)

Superpose sur une même figure l'erreur des cinq méthodes en fonction de `h`
(`:forward`, `:central`, `:complex` en courbes ; `:dual`, `:forward_ad` en traits
horizontaux). Renvoie la figure.
"""
function comparer(; f, dfv, x, v, precision::Type = Float64,
                  hs = 10.0 .^ range(-1, -16, length = 120))
    plt = plot(xscale = :log10, yscale = :log10, legend = :outertopright)
    for m in (:forward, :central, :complex)
        analyse(m; f, dfv, x, v, precision, hs, plt, showbound = false)
    end
    for m in (:dual, :forward_ad)
        analyse(m; f, dfv, x, v, precision, hs, plt)
    end
    plot!(plt, xlabel = "h", ylabel = "erreur",
          title = "erreur vs h — précision $(precision)", bottom_margin = 5mm)
    return plt
end;

In [ ]:
# fonctions test et dérivée exacte
fun1(x) = cos(x)

# bruit multiplicatif : il affecte les parties réelle ET imaginaire.
# (un bruit additif réel `cos(x) + 1e-8*randn()` ne toucherait que la partie réelle
#  et serait ignoré par le schéma complexe.)
fun2(x) = cos(x) * (1 + 1e-8 * randn())

dfun(x) = -sin(x);

In [ ]:
# 1. étude méthode par méthode sur cos en x0 = π/3
x0 = π/3
plts = Any[]
for m in (:forward, :central, :complex)
    plt, hstar, p = analyse(m; f = fun1, dfv = dfun(x0), x = x0, v = 1.0)
    push!(plts, plot(plt, title = string(m)))
end
plot(plts..., layout = (1, 3), size = (1200, 380))

In [ ]:
# 2. les cinq méthodes sur une même figure
comparer(; f = fun1, dfv = dfun(x0), x = x0, v = 1.0)

In [ ]:
# 3a. grand argument  x1 = 10^6 * π/3  (h* décalé vers la droite, L_f ~ ‖x1‖)
x1 = 1e6 * π/3
display(comparer(; f = fun1, dfv = dfun(x1), x = x1, v = 1.0))

# 3b. évaluation bruitée  (plancher relevé, L_f grand)
comparer(; f = fun2, dfv = dfun(x0), x = x0, v = 1.0)

In [ ]:
# 3c. même étude en précision Float32 : courbes et h* décalés d'environ √(eps32/eps64)
comparer(; f = fun1, dfv = Float32(dfun(x0)), x = Float32(x0), v = 1.0f0, precision = Float32)

### Exercice 3 — différences finies centrées (en fin de TP)

1. Reprendre l'analyse d'erreur de la section « différences finies avant » pour le schéma
   centré : établir la majoration
   $$
     \psi(h) \leq \frac{M_2}{6}\, h^2 + \frac{\mathrm{eps}_\mathrm{mach}\, L_f}{h}.
   $$
2. Montrer que $\psi$ est minimale en
   $h_{*} = \left(3\,\mathrm{eps}_\mathrm{mach}\, L_f / M_2\right)^{1/3}$ et que l'erreur
   optimale est d'ordre $\mathrm{eps}_\mathrm{mach}^{2/3}$.
3. Vérifier numériquement sur $\cos$ en $x_0 = \pi/3$ (cellule ci-dessous) : `analyse`
   trace l'erreur, la borne et $h_*$, et affiche l'ordre observé.
4. Que devient $h_*$ pour $\cos + 10^{-8}\,\mathcal{N}(0,1)$ ? pour $x_1 = 10^6\pi/3$ ?


In [ ]:
# vérification numérique du schéma centré  (cf. exercice 3, question 3)
# utiliser analyse(:central; ...) sur fun1 en x0 = π/3, puis fun2 et x1

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Annexe A — pas complexe et cancellation
</div>

**Pourquoi le plateau ?** Contrairement aux différences finies, le schéma complexe ne
forme aucune différence de quantités voisines. La quantité utile, $\mathrm{Im}(f(x+ihv))$,
vaut $\approx h\, g'(0)$ : elle est *petite*, mais calculée avec une erreur **relative**
$\sim \mathrm{eps}_\mathrm{mach}$, soit une erreur absolue $\sim \mathrm{eps}_\mathrm{mach}\, h\, \|g'\|$.
Après division par $h$, l'erreur d'arrondi vaut $\sim \mathrm{eps}_\mathrm{mach}\, \|g'\|$,
**indépendante de $h$**. Rien ne la fait croître quand $h \to 0$ : l'erreur totale décroît
en $h^2$ (troncature) puis se stabilise au plancher $\sim \mathrm{eps}_\mathrm{mach}$.

**Pourquoi la partie réelle ne pollue-t-elle pas la partie imaginaire ?** Brique de base,
la multiplication complexe :
$$
   (a + ib)(c + id) = \underbrace{(ac - bd)}_{\text{réel}}
                    + i\,\underbrace{(ad + bc)}_{\text{imaginaire}},
$$
avec ici $a, c = \mathcal{O}(1)$ et $b, d = \mathcal{O}(h)$. La partie imaginaire calculée
$\mathrm{fl}(ad + bc)$ a une erreur
$\le u\,(|ad| + |bc|) + u\,|ad + bc| = \mathcal{O}(u\, h)$ : **aucun terme en $u\,|ac|$**,
le « gros » produit $\mathcal{O}(1)\times\mathcal{O}(1)$ reste confiné à la partie réelle.
Par récurrence sur le graphe de calcul (chaque nœud $z = \alpha + i\beta$ avec
$\beta = \mathcal{O}(h)$), l'erreur sur $\beta$ reste $\lesssim u\,|\beta|$ : la partie
imaginaire garde une précision *relative* $\mathcal{O}(u)$.

Comme $\mathrm{Re}$ et $\mathrm{Im}$ sont stockés séparément, $x_j + i\,h v_j$ représente
$h v_j$ à $\mathrm{eps}_\mathrm{mach}$ près **même si $h v_j \ll x_j$** — il n'y a jamais
d'addition $x_j + h v_j$ qui écrase $h v_j$. Le pas peut donc descendre jusqu'à
l'underflow.

**Références.**

- W. Squire, G. Trapp, *Using complex variables to estimate derivatives of real
  functions*, SIAM Review 40(1), 1998.
- J. Martins, P. Sturdza, J. Alonso, *The complex-step derivative approximation*,
  ACM TOMS 29(3), 2003.
- N. Higham, *Accuracy and Stability of Numerical Algorithms*, 2e éd., SIAM, 2002,
  §3.6 (arithmétique complexe en virgule flottante).

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Aller plus loin
</div>

### Le pas complexe peut être arbitrairement petit

Pour les différences finies, prendre $h$ trop petit dégrade la dérivée (l'erreur remonte).
Le schéma complexe n'a pas ce défaut : on peut prendre $h$ aussi petit que l'on veut
jusqu'à l'underflow. La cellule ci-dessous le vérifie sur $\cos$ en $\pi/3$.

In [ ]:
# différences avant vs pas complexe quand h → 0
x0 = π/3
println("       h          avant          complexe")
for h in (1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12, 1e-16, 1e-40, 1e-100, 1e-200)
    e_fwd = abs(derivative(cos, x0, 1.0; method = :forward, h = h) - dfun(x0))
    e_cpx = abs(derivative(cos, x0, 1.0; method = :complex, h = h) - dfun(x0))
    @printf("  %.0e      %.3e      %.3e\n", h, e_fwd, e_cpx)
end

### Limites de la différentiation complexe : l'holomorphie

Le pas complexe suppose que **tout** le code de $f$ est bâti sur des opérations qui sont
la vraie extension holomorphe de leur version réelle. C'est le cas de `+ - * /`, `^`,
`exp`, `log`, `sin`, `cos`, `sqrt` (hors demi-axe réel négatif)… mais **pas** de :

- `abs` : sur $\mathbb{C}$, `Julia` calcule le **module** $|z| = \sqrt{\mathrm{re}^2 + \mathrm{im}^2}$,
  fonction à valeurs réelles, non holomorphe — pas l'extension de $x \mapsto |x|$. Donc
  $\mathrm{abs}(2 + ih) = \sqrt{4 + h^2} \approx 2 + h^2/4$, de partie imaginaire nulle :
  le schéma renvoie $0$ au lieu de $1$.
- `min`, `max`, `<`, `>` : pas d'ordre sur $\mathbb{C}$ → le code plante.
- tout code par morceaux (`x ≥ 0 ? … : …`) : idem.

Les nombres duaux et `ForwardDiff`, eux, portent la règle de dérivée de `abs` et donnent
la bonne valeur.

In [ ]:
# f(x) = |x| en x = 2 :  dérivée exacte = 1
for m in methods
    val = try
        derivative(abs, 2.0, 1.0; method = m)
    catch e
        "erreur ($(typeof(e)))"
    end
    @printf("  %-11s : %s\n", m, val)
end

### Dérivée seconde

$\varepsilon^2 = 0$ interdit aux nombres duaux de fournir $f''$ directement. `ForwardDiff`
s'en sort en **emboîtant** deux niveaux de différentiation automatique ; `DualNumbers`
échoue si on tente de composer deux `Dual`.

In [ ]:
# f''(x0) par ForwardDiff imbriqué  (ici f = cos, f'' = -cos)
a  = 0.7
d2 = ForwardDiff.derivative(x -> ForwardDiff.derivative(cos, x), a)
@printf("ForwardDiff imbriqué : f''(%.1f) = %.6f   (exact %.6f)\n", a, d2, -cos(a))

# tentative avec DualNumbers : composer deux Dual échoue
try
    dualpart(dualpart(cos(Dual(Dual(a, 1.0), Dual(1.0, 0.0)))))
catch e
    println("DualNumbers imbriqué : échoue — ", typeof(e))
end